# Sepsis detection - Multivariate Time-Series Classification problem

In [1]:
! python3 -m pip install pandas seaborn numpy matplotlib scikit-learn xgboost scipy "spacy==3.7.5" imbalanced-learn tensorflow

Defaulting to user installation because normal site-packages is not writeable


## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense

from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

/Users/tommy/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Load the dataset and inspects few infromations of the dataset

In [ ]:
dataset = pd.read_csv("./Dataset.csv") # load dataset

In [ ]:
dataset.head(5) # view some rows

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [ ]:
dataset.columns # dataset columns

Index(['Unnamed: 0', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP',
       'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2',
       'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine',
       'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate',
       'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC',
       'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2',
       'HospAdmTime', 'ICULOS', 'SepsisLabel', 'Patient_ID'],
      dtype='object')

In [6]:
patientLabels = dataset.groupby("Patient_ID")["SepsisLabel"].max()
trainData, testData = train_test_split(
    patientLabels.index.to_numpy(), test_size=0.30,
    random_state=67, stratify=patientLabels.to_numpy(),
)
trainData = dataset.loc[dataset["Patient_ID"].isin(trainData)].copy()
testData = dataset.loc[dataset["Patient_ID"].isin(testData)].copy()


## Preprocessing the data : add padding, drop, split, input and scale

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

### Defining a preprocessor to preprocess data

For numerical features, it will be impute using constant value to avoid leakage and then scale using standard scaler

For categorical features, it will also impute using constant and label encoded it

In [ ]:
# transforming pipeline for numerical features
numerical_transformer = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="constant", fill_value= 0)), ("scaler", StandardScaler())]
)

# transforming pipeline for catergorical features
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value = "UNKNOWN")), # no data leak hehe
        ("encoder", LabelEncoder()),
    ]
)

# Extract out catergorical and numerical columns
cat_cols = []
num_cols = []
for col_name, dtype in dataset.dtypes.items():

    if col_name == "SepsisLabel" or col_name == "Patient_ID":
        continue  # SKIP the label and id of the patient

    if np.issubdtype(dtype, np.number):
        num_cols.append(col_name)
    else:
        cat_cols.append(col_name)


# define precessors
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_transformer,
            num_cols
        ),
        (
            "cat",
            categorical_transformer,
            cat_cols,
        ),
        
    ],
    remainder="passthrough" # for the label and id
)

columns = dataset.columns

trainProcessed = preprocessor.fit_transform(trainData)
testProcessed = preprocessor.transform(testData)
dataset_reconstruct = pd.DataFrame(np.vstack([trainProcessed, testProcessed]), columns=columns)


In [ ]:
dataset_reconstruct.shape # correct dimensions

(1552210, 44)

### Time-series processing: 

1. Find 95% of patients window lenght (How many hours 95% percentile patients data last)

2. Drop all patients above that. For patients who above 95%, a sliding window technique can be apply where it is expensive but less expensive to train (trade-off).

3. Transform the problem to detection problem. As a patient can have Sepsis in multiple hours but not others. If there is sepsis in any hour, the patient is classified as sepsis

NOTE: justification to remain time-series problem rather time-step prediction ( simple supervised problem ): temporal information can carries informations about the patient information hence having full window retains useful information

In [ ]:
groups = dataset_reconstruct.groupby("Patient_ID") # Group patients records

hours =[] # how many hours is each patient records last

transformed_sequence = []

# Analysis of the patient time lentgh
for patient_id, patient_df in groups:
    hours.append(patient_df.shape[0])

window = np.percentile(hours,95) # 95% percentile
window # 58 hours


58.0

In [11]:
# preprocessing : drop all > 95% percentile hours and add padding
target_var = []


for patient_id, patient_df in groups:

    #extract the target variable
    # if the patient have sepsis, they will have at least 1 row to be 1
    label = 0
    for i in patient_df["SepsisLabel"]:
        if i == 1:
            label = 1

    patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])

    if patient_df.shape[0] > window:
        #im lazy sorry:(
        continue

    target_var.append(label)
    #add padding
    patient_df = patient_df.sort_values("Hour")
    
    num_feature = patient_df.shape[1] # should be 43 anyway


    #change to np
    patient_np = patient_df.to_numpy()
    num_padding_rows = int(window) - patient_df.shape[0]
    all_zeros = np.zeros(num_feature)
    new_patient = []
    for i in range (num_padding_rows):
        new_patient.append(all_zeros)
    for hour in patient_np:
        new_patient.append(hour)

    transformed_sequence.append(new_patient)

transformed_sequence = np.asarray(transformed_sequence)
print(transformed_sequence.shape)

(38617, 58, 42)


In [ ]:
transformed_sequence[0:5] # Expect a 3D array 

array([[[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        ...,
        [ 8.83137308e-01,  8.83137308e-01,  2.91347620e-01, ...,
         -6.62381181e-01,  3.45717042e-01,  8.62080829e-01],
        [ 9.17760293e-01,  9.17760293e-01,  3.24540480e-01, ...,
         -6.62381181e-01,  3.45717042e-01,  8.96557156e-01],
        [ 9.52383277e-01,  9.52383277e-01,  2.58154760e-01, ...,
         -6.62381181e-01,  3.45717042e-01,  9.31033483e-01]],

       [[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
          0.00000000e+00,  0.00000000e

In [ ]:
target_var # this is our ground truth

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,


In [ ]:
target_var_np = np.asarray(target_var) # transform to np array

In [ ]:
target_var_np.shape # check shape

(38617,)

In [ ]:
# Split train test
patientSizes = groups.size()
trainMask = patientSizes[patientSizes <= window].index.isin(trainData["Patient_ID"])
X_train, X_test = transformed_sequence[trainMask], transformed_sequence[~trainMask]
y_train, y_test = target_var_np[trainMask], target_var_np[~trainMask]

print(X_train.shape, X_test.shape)


(27031, 58, 42)
(11586, 58, 42)
27031
11586


In [17]:
print(len([one for one in y_train if one == 0])) # How non-sepsis (Data is VERY IMBALANCE)

25696


### Upsampling by SMOTE
1. Flatten out to SMOTE

2. Apply SMOTE (SMOTE perturbs and use KKN to get the neighbours). This upsampling under-represented label

3. Fit SMOTE and yeah!

In [18]:
#SMOTE sampling X_train

#Flatten to 2D

num_sample, num_time, num_feature = X_train.shape

X_train_flatten = X_train.reshape(num_sample, num_time*num_feature) #Flatten to do SMOTE
#SMOTE time
s = SMOTE(k_neighbors=10, random_state= 67)
X_train_sampled_flatten , y_train_sampled = s.fit_resample(X_train_flatten,y_train)
X_train_sampled = X_train_sampled_flatten.reshape(-1, num_time, num_feature) # reshape back to 3D

X_train_sampled.shape

(51392, 58, 42)

In [19]:
# check shapes
print(len(y_train_sampled))

print(len([one for one in y_train_sampled if one == 1]))
print(len([one for one in y_train_sampled if one == 0]))

# data is balanced now

51392
25696
25696


In [20]:
# recontruct back to np array
X_train_sampled = np.asarray(X_train_sampled, dtype=np.float32)
y_train_sampled = np.asarray(y_train_sampled, dtype=np.float32)
X_test = np.asarray(X_test, dtype=np.float32)
y_test = np.asarray(y_test, dtype=np.float32)

### Baseline: RNN (simple small) - Note: No pass this baseline on Recall Sepsis -> Try again

In [ ]:
# Baseline SimpleRNN for the patient sequences
import random
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

SEED = 67
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

num_time_steps, num_features = X_train_sampled.shape[1], X_train_sampled.shape[2]

model = Sequential([
    SimpleRNN(32, input_shape=(num_time_steps, num_features)),
    Dense(1, activation="sigmoid"),
])
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
model.fit(
    X_train_sampled,
    y_train_sampled,
    epochs=10,
    batch_size=100,
    verbose=2,
)



Epoch 1/10


C:\Users\Samuel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


514/514 - 2s - 4ms/step - accuracy: 0.8683 - loss: 0.3129
Epoch 2/10
514/514 - 1s - 3ms/step - accuracy: 0.9166 - loss: 0.2165
Epoch 3/10
514/514 - 1s - 3ms/step - accuracy: 0.9324 - loss: 0.1780
Epoch 4/10
514/514 - 1s - 3ms/step - accuracy: 0.9395 - loss: 0.1597
Epoch 5/10
514/514 - 2s - 3ms/step - accuracy: 0.9452 - loss: 0.1456
Epoch 6/10
514/514 - 2s - 3ms/step - accuracy: 0.9494 - loss: 0.1371
Epoch 7/10
514/514 - 1s - 3ms/step - accuracy: 0.9535 - loss: 0.1243
Epoch 8/10
514/514 - 2s - 3ms/step - accuracy: 0.9563 - loss: 0.1194
Epoch 9/10
514/514 - 2s - 4ms/step - accuracy: 0.9585 - loss: 0.1126
Epoch 10/10
514/514 - 2s - 4ms/step - accuracy: 0.9592 - loss: 0.1125
              precision    recall  f1-score   support

   No sepsis     0.9801    0.9513    0.9655     11024
      Sepsis     0.4033    0.6302    0.4919       576

    accuracy                         0.9353     11600
   macro avg     0.6917    0.7907    0.7287     11600
weighted avg     0.9515    0.9353    0.9420     

GRU

In [51]:
from tensorflow.keras.layers import GRU

model =tf.keras.Sequential()
model.add(GRU(16, input_shape=X_train_sampled.shape[1:])) 
model.add(Dense(1, activation="sigmoid")) 
model.add(Dense(1, activation="sigmoid")) 

model.compile(loss='binary_crossentropy', optimizer='adam')
model.fit(X_train, y_train, class_weight= {0:0.1,1:5}, epochs=20, batch_size=100, verbose=2)



Epoch 1/20


C:\Users\Samuel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


271/271 - 2s - 9ms/step - loss: 0.2336
Epoch 2/20
271/271 - 2s - 6ms/step - loss: 0.2059
Epoch 3/20
271/271 - 2s - 6ms/step - loss: 0.1896
Epoch 4/20
271/271 - 2s - 6ms/step - loss: 0.1773
Epoch 5/20
271/271 - 2s - 6ms/step - loss: 0.1668
Epoch 6/20
271/271 - 2s - 6ms/step - loss: 0.1581
Epoch 7/20
271/271 - 1s - 6ms/step - loss: 0.1504
Epoch 8/20
271/271 - 2s - 6ms/step - loss: 0.1436
Epoch 9/20
271/271 - 2s - 6ms/step - loss: 0.1380
Epoch 10/20
271/271 - 2s - 6ms/step - loss: 0.1331
Epoch 11/20
271/271 - 2s - 6ms/step - loss: 0.1288
Epoch 12/20
271/271 - 2s - 6ms/step - loss: 0.1249
Epoch 13/20
271/271 - 2s - 6ms/step - loss: 0.1222
Epoch 14/20
271/271 - 2s - 6ms/step - loss: 0.1183
Epoch 15/20
271/271 - 2s - 6ms/step - loss: 0.1154
Epoch 16/20
271/271 - 2s - 6ms/step - loss: 0.1139
Epoch 17/20
271/271 - 2s - 6ms/step - loss: 0.1114
Epoch 18/20
271/271 - 2s - 6ms/step - loss: 0.1090
Epoch 19/20
271/271 - 2s - 6ms/step - loss: 0.1069
Epoch 20/20
271/271 - 2s - 6ms/step - loss: 0.1061


In [52]:
test_probability = model.predict(X_test, verbose=0).ravel()
test_prediction = (test_probability >= 0.5).astype(np.int32)
print(classification_report(
    y_test,
    test_prediction,
    target_names=["No sepsis", "Sepsis"],
    digits=4,
    zero_division=0,
))
print(f"ROC-AUC: {roc_auc_score(y_test, test_probability):.4f}")

              precision    recall  f1-score   support

   No sepsis     0.9898    0.8435    0.9108     11024
      Sepsis     0.2177    0.8333    0.3452       576

    accuracy                         0.8430     11600
   macro avg     0.6037    0.8384    0.6280     11600
weighted avg     0.9514    0.8430    0.8827     11600

ROC-AUC: 0.9002


In [31]:
import joblib
model.save("./baseline_rnn.keras")
joblib.dump(preprocessor, "./preprocessor")

['./preprocessor']

In [32]:
# Load the saved model and preprocessing pipeline
import joblib
import tensorflow as tf

model = tf.keras.models.load_model("./baseline_rnn.keras")
preprocessor = joblib.load("./preprocessor")

In [33]:
dataset_reconstruct

,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,-0.876087,-0.876087,-2.534103,-2.572607,-0.713839,-2.176542,-2.314090,-1.383820,-1.928192,-0.19124,...,-0.072074,-0.220801,0.403300,-1.131184,-0.658136,-0.658014,0.338431,-0.888430,0.0,17072.0
1,-0.841829,-0.841829,-0.371529,0.471241,-0.713839,-2.176542,-0.007806,-1.383820,0.077844,-0.19124,...,-0.072074,-0.220801,0.403300,-1.131184,-0.658136,-0.658014,0.338431,-0.854378,0.0,17072.0
2,-0.807572,-0.807572,0.060986,0.471241,-0.713839,-2.176542,-0.952742,-1.383820,-1.928192,-0.19124,...,-0.072074,-0.220801,0.403300,-1.131184,-0.658136,-0.658014,0.338431,-0.820326,0.0,17072.0
3,-0.773314,-0.773314,-0.105365,0.471241,-0.713839,-2.176542,-2.314090,-1.383820,0.138633,-0.19124,...,-0.072074,-0.220801,0.403300,-1.131184,-0.658136,-0.658014,0.338431,-0.786275,0.0,17072.0
4,-0.739056,-0.739056,-0.205177,0.471241,-0.713839,0.475121,0.056257,0.786198,-0.226100,-0.19124,...,-0.072074,6.039402,0.403300,-1.131184,-0.658136,-0.658014,0.338431,-0.752223,0.0,17072.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1552205,-0.430738,-0.430738,-2.534103,-2.572607,-0.713839,-2.176542,-2.314090,-1.383820,-1.928192,-0.19124,...,-0.072074,-0.220801,0.187918,-1.131184,-0.658136,-0.658014,0.313549,-0.445760,0.0,105149.0
1552206,-0.396480,-0.396480,-2.534103,-2.572607,-0.713839,-2.176542,-2.314090,-1.383820,0.503367,-0.19124,...,-0.072074,-0.220801,0.187918,-1.131184,-0.658136,-0.658014,0.313549,-0.411708,0.0,105149.0
1552207,-0.362223,-0.362223,-2.534103,-2.572607,-0.713839,-2.176542,-2.314090,-1.383820,-1.928192,-0.19124,...,-0.072074,-0.220801,0.187918,-1.131184,-0.658136,-0.658014,0.313549,-0.377656,0.0,105149.0
1552208,-0.327965,-0.327965,-2.534103,-2.572607,-0.713839,-2.176542,-2.314090,-1.383820,-1.928192,-0.19124,...,-0.072074,-0.220801,0.187918,-1.131184,-0.658136,-0.658014,0.313549,-0.343605,0.0,105149.0


In [34]:
dummy_num_rows = 10
# The saved preprocessor was fitted with all original columns, so use zero placeholders
# for the excluded columns only while transforming, then keep them for add_padding.
dummy_raw = pd.DataFrame(
    np.zeros((dummy_num_rows, len(dataset.columns))),
    columns=dataset.columns,
)
dummy_data = pd.DataFrame(
    preprocessor.transform(dummy_raw),
    columns=dataset.columns,
)

print(f"Dummy patient shape after preprocessing: {dummy_data.shape}")

def add_padding(patient_df, window):
    # Assumes that the patient has fewer rows than the window.
    for i in patient_df["SepsisLabel"]:
        print(i)
    patient_df = patient_df.drop(columns=["Patient_ID", "SepsisLabel"])
    patient_df = patient_df.sort_values("Hour")

    num_feature = patient_df.shape[1]
    patient_np = patient_df.to_numpy()
    num_padding_rows = int(window) - patient_df.shape[0]
    all_zeros = np.zeros(num_feature)
    new_patient = []
    count = 0
    for i in range(num_padding_rows):
        new_patient.append(all_zeros)
        count += 1
    for hour in patient_np:
        new_patient.append(hour)
        count += 1
    new_patient_np = np.asarray(new_patient)
    print(count)
    print(len(new_patient))
    return new_patient_np

X_infer = add_padding(dummy_data, 58)[None, ...]
print(X_infer.shape)
y = model.predict(X_infer)
if y[0][0] > 0.5:
    print("Sepsis")

if y[0][0] < 0.5:
    print("No Sepsis")
print(y)

Dummy patient shape after preprocessing: (10, 44)
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
58
58
(1, 58, 42)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
No Sepsis
[[0.10564511]]
